# Data Cleaning
Fixing missing values, impossible rows and data leakage before analysis.

In [1]:
import pandas as pd

df = pd.read_csv("../data/hotel_bookings.csv")
print("Rows, columns before cleaning:", df.shape)

Rows, columns before cleaning: (119390, 32)


In [2]:
df = df.fillna({
    "children": 0,
    "country": "Unknown",
    "agent": 0,
    "company": 0,
})
df["children"] = df["children"].astype(int)

print("Missing values left:", df.isna().sum().sum())

Missing values left: 0


In [3]:
no_guests = (df["adults"] + df["children"] + df["babies"]) == 0
bad_price = (df["adr"] < 0) | (df["adr"] > 1000)

print("Bookings with 0 guests:", no_guests.sum())
print("Bookings with impossible price:", bad_price.sum())

df = df[~(no_guests | bad_price)]
print("Rows, columns after removing them:", df.shape)

Bookings with 0 guests: 180
Bookings with impossible price: 2
Rows, columns after removing them: (119208, 32)


In [4]:
pd.crosstab(df["reservation_status"], df["is_canceled"])

is_canceled,0,1
reservation_status,,
Canceled,0,42992
Check-Out,75010,0
No-Show,0,1206


In [5]:
df = df.drop(columns=["reservation_status", "reservation_status_date"])
print("Rows, columns after dropping leaky columns:", df.shape)

Rows, columns after dropping leaky columns: (119208, 30)


In [6]:
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

df[["stays_in_weekend_nights", "stays_in_week_nights", "total_nights",
    "adults", "children", "babies", "total_guests"]].head()

,stays_in_weekend_nights,stays_in_week_nights,total_nights,adults,children,babies,total_guests
0,0,0,0,2,0,0,2
1,0,0,0,2,0,0,2
2,0,1,1,1,0,0,1
3,0,1,1,1,0,0,1
4,0,2,2,2,0,0,2


In [7]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 32238


In [8]:
df.to_csv("../data/hotel_bookings_clean.csv", index=False)
print("Saved clean data:", df.shape)

Saved clean data: (119208, 32)


## Cleaning summary
| Problem | Action | Reason |
|---|---|---|
| Missing `children` (4), `country` (488) | Filled with 0 / "Unknown" | Too few to matter; can't guess a country |
| Missing `agent`, `company` | Filled with 0 | Missing means "no agent / no company" |
| 0 guests (180 rows) | Removed | A booking for nobody is impossible |
| Price < 0 or > 1000 (2 rows) | Removed | Data-entry errors |
| `reservation_status`, `reservation_status_date` | Dropped | Data leakage: only known after the outcome |
| 32,238 identical rows | Kept (limitation) | No booking ID; group bookings look identical |
| — | Added `total_nights`, `total_guests` | Easier analysis |

**Result:** 119,390 → 119,208 rows